# Phase 6 (post hoc): P18's frozen pipeline on 1,000 unused Icentia11k patients (phase6_fresh_patients, P19)

PREREGISTRATION §10 v18, P19. The first segment (about 70 minutes) of 1,000 Icentia11k patients drawn at random (seed 1)
from the 10,600 that P14 did not draw; every usable N, S and V beat in P13's form. P18's sixty runs, P16's AF detector,
threshold and form, and Huang et al.'s threshold are used as they are; nothing is fitted or chosen, and DS2 is not read.

| `PART` | accelerator | inputs | what it does |
|---|---|---|---|
| `run` | GPU | `ecgaug-repo`, `ecgaug-p13-runs`, `ecgaug-p15-runs`, `ecgaug-p18-runs` | downloads and builds the beats (all CPU cores), predicts with the 60 runs on the GPU, scores |

`build` and `evaluate` split the same work in two. Settings: **Internet on** (PhysioNet's open-data mirror).
Download the zip in Output and run `python kaggle/ingest_runs.py <zip>`.


In [ ]:
# --- Parameters: edit only this cell ---------------------------------------------------------
PART = "run"                          # "run" (GPU): build, predict, score; or "build" / "evaluate"
RUN_TESTS = False                     # the full test suite first; nothing else runs if it fails
INPUTS_DIR = "/kaggle/input"          # attached datasets: banks, checkpoints (runs/), external data (data/)
GIT_REF = "main"                      # branch name or full commit SHA to run
DEVICE = "auto"                       # "auto" uses the GPU when the session has one
ENV_MODE = "locked"                   # "locked" = exact uv.lock versions; "kaggle" = preinstalled packages (faster, debugging only)
RAW_DATASET_DIR = "/kaggle/input"     # attached mitdb-raw files; if none, downloaded from PhysioNet's AWS mirror (checksums checked)
REPO_URL = "https://github.com/akashsaha477/Gen_Ai_Project.git"


In [ ]:
import os, subprocess, sys
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = WORK / "repo"
env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    token = None                      # public repository, or the secret is not attached
if token:
    askpass = WORK / ".git-askpass.sh"
    askpass.write_text('#!/bin/sh\necho "$GITHUB_TOKEN"\n')
    askpass.chmod(0o700)
    env.update(GIT_ASKPASS=str(askpass), GITHUB_TOKEN=token)   # the token is never printed or put in a URL
    url = REPO_URL.replace("https://", "https://x-access-token@")

if not REPO.exists():
    subprocess.run(["git", "clone", "--quiet", url, str(REPO)], check=True, env=env)
subprocess.run(["git", "fetch", "--quiet", "origin"], cwd=REPO, check=True, env=env)
ref = GIT_REF if len(GIT_REF) == 40 else f"origin/{GIT_REF}"
subprocess.run(["git", "checkout", "--quiet", "--detach", ref], cwd=REPO, check=True)
sha = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO, capture_output=True, text=True).stdout.strip()
print("running commit", sha)

In [ ]:
configs = ["configs/phase6_fresh_patients/fresh_patients.yaml"]
extra = ["--part", PART, "--device", DEVICE]
cmd = [sys.executable, "kaggle/kaggle_run.py", "--cli", "ecgaug.experiments.phase6_fresh_patients", "--env", ENV_MODE, "--out", str(WORK),
       "--config", *configs, "--raw-dir", RAW_DATASET_DIR, "--artifacts-dir", INPUTS_DIR, "--data-dir", INPUTS_DIR]
if RUN_TESTS:
    cmd += ["--tests"]
cmd += ["--", *extra]
code = subprocess.run(cmd, cwd=REPO, env=env).returncode
print("finished with exit code", code, "(2 = the run was refused, read the message above)")
